<a href="https://colab.research.google.com/github/loomfox/Houston-Election-Util/blob/main/coh_canvass_2010_present_parser_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COH Election Canvass Scraper & Multi-Format Parser (2010–present)

Downloads every City of Houston City Secretary canvass/recap PDF from
2010-11-02 onward, **fingerprints each PDF's layout**, routes it to a
**saved parser function for that specific format**, and assembles the
results into per-election DataFrames plus one combined long-format dataset.

**Why "multi-format" matters:** COH's canvass PDFs are not produced by one
consistent system across 16 years. Column layout, header wording, and
even whether the PDF is real text vs. a scan have drifted over time, and
the city's own index page has at least one broken link (a missing `.` in
a filename), which is itself a small signal that format/consistency isn't
guaranteed. Rather than writing one regex and hoping, this notebook:

1. Downloads each PDF.
2. **Fingerprints** it by extracting page-1 text and testing it against a
   registry of known header signatures.
3. Routes it to the **named parser function saved for that format**
   (e.g. `parse_format_tabular_ballot_breakdown`).
4. Logs anything that matches no known format to an `unrecognized` manifest
   instead of crashing or silently mis-parsing — so you can inspect it and
   write a new parser function for that one format without touching the rest.
5. Builds one DataFrame **per election** (`election_dfs[election_id]`) plus
   a combined `df_all` across every election that parsed successfully.

**Scope:** 2010-11-02 through the most recent election on the live index
page as of whenever you run this (it re-scrapes the index live each run,
so newer elections get picked up automatically).

**Runtime:** pure CPU, text/regex based — no GPU needed, runs fine on
standard Colab.


## 1. Install dependencies


In [ ]:
!pip install -q pdfplumber requests pandas openpyxl


## 2. Config

`MIN_DATE` is inclusive. The index page mixes a left column ("Canvass of
General and Run-off Elections", sparse beyond the 1980s) and a right column
("Recap of General and Run-off Elections", which is where the 2010+ links
actually live). The scraper in Section 3 pulls every `.pdf` link from the
whole page and filters by date, so it doesn't matter which column an entry
sits in.


In [ ]:
import os
import re
import io
import json
import time
import datetime as dt
import pandas as pd

INDEX_URL = "https://houstontx.gov/citysec/elections/electionsindex.html"
BASE_URL = "https://houstontx.gov/citysec/elections/"

MIN_DATE = dt.date(2010, 1, 1)   # inclusive lower bound; adjust if needed
TODAY = dt.date.today()

WORKDIR = "/content/coh_canvass"
RAW_DIR = os.path.join(WORKDIR, "raw_pdfs")
MANIFEST_PATH = os.path.join(WORKDIR, "manifest.json")
UNRECOGNIZED_DIR = os.path.join(WORKDIR, "unrecognized_samples")

for d in (WORKDIR, RAW_DIR, UNRECOGNIZED_DIR):
    os.makedirs(d, exist_ok=True)

print(f"Scoping to elections on/after {MIN_DATE.isoformat()}")
print(f"Working directory: {WORKDIR}")


## 3. Scrape the index page for every PDF link, filtered to scope

Filenames follow `MMDDYY.pdf` (e.g. `110811.pdf` = Nov 8, 2011). A handful
of two-digit years are ambiguous past 2030, but since this notebook is
scoped to historical Houston municipal elections, any `YY <= 50` is treated
as `20YY` and anything else as `19YY` — safe for the foreseeable future of
this dataset.

The live page has at least one known broken link (`121419pdf`, missing the
dot before `pdf`). The scraper repairs that specific pattern automatically;
any other malformed link found in the future gets logged to `broken_links`
in the manifest rather than silently dropped, so you'll see it printed
below if the city breaks something else later.


In [ ]:
import requests
from html.parser import HTMLParser

class LinkExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []
    def handle_starttag(self, tag, attrs):
        if tag == "a":
            for name, value in attrs:
                if name == "href" and value:
                    self.links.append(value)

def fetch_index_html():
    """Fetch the index page with retry/backoff; some cloud IP ranges have
    had trouble reaching houstontx.gov directly in the past."""
    last_err = None
    for attempt in range(4):
        try:
            resp = requests.get(INDEX_URL, timeout=20, headers={
                "User-Agent": "Mozilla/5.0 (compatible; civic-data-research/1.0)"
            })
            resp.raise_for_status()
            return resp.text
        except Exception as e:
            last_err = e
            wait = 2 ** attempt
            print(f"  fetch attempt {attempt+1} failed ({e}); retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError(
        f"Could not reach {INDEX_URL} after retries ({last_err}). "
        "If this persists, the Colab IP range may be blocked -- try "
        "downloading electionsindex.html manually and set MANUAL_INDEX_HTML_PATH below."
    )

MANUAL_INDEX_HTML_PATH = None  # set to a local path if fetch_index_html() fails

if MANUAL_INDEX_HTML_PATH:
    with open(MANUAL_INDEX_HTML_PATH, "r", encoding="utf-8") as f:
        index_html = f.read()
else:
    index_html = fetch_index_html()

parser = LinkExtractor()
parser.feed(index_html)

DATE_RE = re.compile(r"^(\d{2})(\d{2})(\d{2})\.?pdf$", re.IGNORECASE)

candidates = []
broken_links = []
for href in parser.links:
    fname = href.rsplit("/", 1)[-1]
    if not fname.lower().endswith("pdf"):
        continue
    m = DATE_RE.match(fname)
    if not m:
        broken_links.append(href)
        continue
    mm, dd, yy = m.groups()
    year = int(yy)
    year += 2000 if year <= 50 else 1900
    try:
        election_date = dt.date(year, int(mm), int(dd))
    except ValueError:
        broken_links.append(href)
        continue

    # repair the known missing-dot pattern (e.g. "121419pdf" -> "121419.pdf")
    clean_fname = f"{mm}{dd}{yy}.pdf"
    full_url = BASE_URL + clean_fname
    election_id = f"{mm}{dd}{yy}"

    candidates.append({
        "election_id": election_id,
        "election_date": election_date.isoformat(),
        "election_year": year,
        "url": full_url,
        "source_href": href,
    })

# de-dupe by election_id (left/right columns or repeated links can duplicate entries)
seen = {}
for c in candidates:
    seen[c["election_id"]] = c
candidates = sorted(seen.values(), key=lambda c: c["election_date"])

in_scope = [c for c in candidates if dt.date.fromisoformat(c["election_date"]) >= MIN_DATE]

print(f"Found {len(candidates)} total dated PDF links on the index page.")
print(f"{len(in_scope)} are in scope ({MIN_DATE.isoformat()} onward).")
if broken_links:
    print(f"\n{len(broken_links)} link(s) did not match the expected MMDDYY(.)pdf pattern:")
    for b in broken_links:
        print(f"  - {b}")
    print("These are skipped automatically. Inspect manually if any look like they should be in scope.")

for c in in_scope:
    print(f"  {c['election_id']}  ({c['election_date']})  -> {c['url']}")


## 4. Download PDFs

Skips re-downloading anything already on disk, so this cell is safe to
re-run if a connection drops partway through.


In [ ]:
def local_pdf_path(election_id):
    return os.path.join(RAW_DIR, f"{election_id}.pdf")

download_log = []

for c in in_scope:
    path = local_pdf_path(c["election_id"])
    if os.path.exists(path) and os.path.getsize(path) > 0:
        download_log.append({**c, "status": "already_downloaded"})
        continue
    try:
        resp = requests.get(c["url"], timeout=30, headers={
            "User-Agent": "Mozilla/5.0 (compatible; civic-data-research/1.0)"
        })
        resp.raise_for_status()
        with open(path, "wb") as f:
            f.write(resp.content)
        download_log.append({**c, "status": "downloaded", "bytes": len(resp.content)})
        print(f"  downloaded {c['election_id']}.pdf ({len(resp.content):,} bytes)")
    except Exception as e:
        download_log.append({**c, "status": "download_failed", "error": str(e)})
        print(f"  FAILED {c['election_id']}: {e}")
    time.sleep(0.5)  # light courtesy delay

failed = [d for d in download_log if d["status"] == "download_failed"]
print(f"\n{len(download_log) - len(failed)} / {len(download_log)} PDFs available locally.")
if failed:
    print("Failed downloads (re-run this cell to retry):")
    for f_ in failed:
        print(f"  - {f_['election_id']}: {f_['error']}")


## 4b. Diagnostic: spot-check real page text (optional, but useful if Section 5b looks wrong)

The section-detection patterns in Section 5 (`SECTION_HEADER_PATTERNS`,
`OFFICE_LINE_RE`) and the parsers in Section 5d were built and tested
against real extracted text from a 2019 mayoral canvass PDF, not guessed
from search snippets -- an earlier version of this notebook guessed from
snippets and it didn't work, which is why this diagnostic cell exists at
all.

That said, **a different election's PDF could still use a different
header wording**, especially older PDFs or special/runoff elections that
weren't part of what was checked. If Section 5b's printed run table looks
wrong for some election -- offices not splitting where they should, or
everything landing in `UNCLASSIFIED` -- run this cell for that specific
election and compare what it prints against what `SECTION_HEADER_PATTERNS`
and `OFFICE_LINE_RE` expect, rather than guessing at a fix.


In [ ]:
import pdfplumber

# sample a handful of PDFs spread across the scoped range so we see early,
# middle, and recent eras in one pass
successful_ids = [d["election_id"] for d in download_log if d["status"] in ("downloaded", "already_downloaded")]
if successful_ids:
    sample_ids = sorted(set([
        successful_ids[0],
        successful_ids[len(successful_ids) // 4],
        successful_ids[len(successful_ids) // 2],
        successful_ids[(3 * len(successful_ids)) // 4],
        successful_ids[-1],
    ]))
else:
    sample_ids = []

PAGES_TO_SHOW = 2  # page 1 (often summary) and page 2 (often precinct-level)
CHARS_PER_PAGE = 1500  # truncate long pages so output stays readable

for election_id in sample_ids:
    path = local_pdf_path(election_id)
    print("=" * 80)
    print(f"ELECTION {election_id}")
    print("=" * 80)
    try:
        with pdfplumber.open(path) as pdf:
            n_pages = len(pdf.pages)
            print(f"({n_pages} total pages in this PDF)")
            for page_idx in range(min(PAGES_TO_SHOW, n_pages)):
                text = pdf.pages[page_idx].extract_text() or "<NO EXTRACTABLE TEXT -- LIKELY SCANNED/IMAGE-ONLY>"
                print(f"\n--- page {page_idx + 1} ---")
                print(text[:CHARS_PER_PAGE])
                if len(text) > CHARS_PER_PAGE:
                    print(f"... [truncated, {len(text)} total chars on this page]")
    except Exception as e:
        print(f"COULD NOT OPEN: {e}")
    print()


## 5. Section detection (confirm before parsing)

Real page text from a 2019 canvass PDF (1,012-precinct mayoral race,
317 physical pages) revealed two things the earlier blind-fingerprint
approach got wrong:

1. **Page 1 of the PDF is not page 1 of any logical section.** The
   document opens with a multi-page **"Combined Totals" summary block**
   (citywide office-by-office results, immediately followed by the same
   offices split out by Harris/Fort Bend/Montgomery county), and only
   *after* that does the **"Canvass Report — Total Voters — Official"**
   precinct-level section begin — which then itself runs for hundreds of
   pages, with **a new sub-section starting every time the office changes**
   (Mayor's table, then Controller, then Council District A, etc.), each
   restarting its own internal page count ("Page 1 of 317" is the
   document's physical page 5, not the canvass section's start).

2. **Column order is not what was guessed earlier.** It is
   `Election Ballots Cast | Total Ballots Cast | Registered Voters |
   Percent Turnout | <candidate name columns...> | Early Ballots Cast` —
   Early Ballots Cast is the **last** column, not the second. A parser
   built on the earlier guess would have silently mislabeled every vote
   count.

Rather than write parsers against one sample and hope they generalize,
this section:

- Walks every page of a PDF and computes a **section key** per page —
  enough identifying text (header phrase + current office line) to tell
  "this page continues the same section as the last page" from "this page
  starts a new section."
- Groups consecutive pages sharing a section key into **section runs**
  (e.g. "Mayor canvass table, pages 5–45").
- Prints a **summary table of detected section runs** for you to read and
  confirm or correct — this is the human-in-the-loop checkpoint your
  workflow called for, done here as a printed confirmation list (Colab's
  default widgets aren't rich enough for a true interactive prompt, so
  correction happens by editing the `SECTION_OVERRIDES` dict in the next
  cell and re-running).
- Once confirmed, samples the **first, middle, and last page** of each
  section run for parser derivation, and flags any two section runs whose
  samples are byte-for-byte identical in structure so you don't end up
  with duplicate parsers doing the same job.


In [ ]:
import pdfplumber

SECTION_HEADER_PATTERNS = [
    # (label, regex) -- checked in order, first match wins for "what kind of
    # page is this". These are taken from REAL extracted text (2019 mayoral
    # canvass PDF), not guessed from search snippets.
    #
    # The combined-totals summary section has two header variants seen in
    # the real PDF: the very first page says "...COUNTIES COMBINED" with
    # no later page number, while every subsequent page in that section
    # says "...Combined Totals Page N". Both must match.
    ("combined_totals_summary", re.compile(r"COUNTIES\s+COMBINED|Combined\s+Totals\s+Page", re.IGNORECASE)),
    ("canvass_report_detail", re.compile(r"Canvass Report\s*[—-]\s*Total Voters", re.IGNORECASE)),
]

OFFICE_LINE_RE = re.compile(
    r"City of Houston,\s*(Mayor|Controller)|"
    r"Council Member,?\s*District\s+[A-K]|"
    r"Council Member,?\s*At[- ]Large Position\s+\d+",
    re.IGNORECASE,
)

def classify_page(text):
    """Returns (header_label, office_line) for one page's extracted text.
    header_label is None if no known header pattern matched (page goes to
    an 'unclassified' bucket rather than crashing)."""
    header_label = None
    for label, pattern in SECTION_HEADER_PATTERNS:
        if pattern.search(text):
            header_label = label
            break

    office_match = OFFICE_LINE_RE.search(text)
    office_line = office_match.group(0).strip() if office_match else None

    return header_label, office_line

def section_key(header_label, office_line):
    """The combined_totals_summary section enumerates many offices across
    its handful of pages -- those pages are all the SAME section type and
    should group into one run, not split per office. The
    canvass_report_detail section, by contrast, genuinely changes table
    structure/content per office and should split into a new run each
    time the office changes (a new office means a new precinct table
    starting back at precinct 0001). So office_line only matters for the
    grouping key when we're inside canvass_report_detail."""
    if header_label == "canvass_report_detail":
        return (header_label, office_line)
    return (header_label, None)

def detect_sections(pdf_path):
    """Walks every page of a PDF and groups consecutive pages sharing the
    same (header_label, office_line) into section runs. Returns a list of
    dicts: {start_page, end_page, header_label, office_line, n_pages}.

    Page numbers in the output are 1-indexed and refer to the PDF's
    PHYSICAL page number, not any internal page numbering printed on the
    page itself (the 2019 PDF resets its own "Page X of 317" counter at
    the start of the canvass detail section, which is a property of the
    document, not something to rely on for navigation).
    """
    runs = []
    current_run = None

    with pdfplumber.open(pdf_path) as pdf:
        for page_idx, page in enumerate(pdf.pages):
            text = page.extract_text() or ""
            header_label, office_line = classify_page(text)
            key = section_key(header_label, office_line)

            if current_run is not None and current_run["key"] == key:
                current_run["end_page"] = page_idx + 1
                current_run["n_pages"] += 1
                # office_line can still change within a combined_totals_summary
                # run (it enumerates many offices) -- track all of them
                if office_line and office_line not in current_run["offices_seen"]:
                    current_run["offices_seen"].append(office_line)
            else:
                if current_run is not None:
                    runs.append(current_run)
                current_run = {
                    "key": key,
                    "header_label": header_label,
                    "office_line": office_line,
                    "offices_seen": [office_line] if office_line else [],
                    "start_page": page_idx + 1,
                    "end_page": page_idx + 1,
                    "n_pages": 1,
                }
        if current_run is not None:
            runs.append(current_run)

    return runs

def print_section_runs(runs, election_id):
    print(f"\nDetected {len(runs)} section run(s) for {election_id}:")
    print(f"{'#':>3}  {'pages':>12}  {'n':>4}  {'header':<24}  offices_seen")
    for i, r in enumerate(runs):
        page_range = f"{r['start_page']}-{r['end_page']}"
        header = r["header_label"] or "UNCLASSIFIED"
        offices = ", ".join(r["offices_seen"][:3])
        if len(r["offices_seen"]) > 3:
            offices += f", ... (+{len(r['offices_seen']) - 3} more)"
        if not offices:
            offices = "-"
        print(f"{i:>3}  {page_range:>12}  {r['n_pages']:>4}  {header:<24}  {offices}")


## 5b. Run section detection across every downloaded PDF, confirm or correct

For each election, prints the detected section runs. **Read this output.**
If a run's `offices_seen` or page range looks wrong — e.g. a section that
should have split into two didn't, or a section absorbed pages that don't
belong to it — note the election_id and what's wrong.

To correct a detected run without editing the detection regexes (useful
for one-off oddities in a specific election), add an entry to
`SECTION_OVERRIDES` below and re-run this cell. To fix something that's
wrong across *every* election (e.g. a header pattern that's too narrow),
go back and edit `SECTION_HEADER_PATTERNS` / `OFFICE_LINE_RE` in Section 5
instead, since that's a detection-logic problem, not a one-off override.


In [ ]:
# Manual override hook: {election_id: [list of corrected run dicts]}.
# Leave empty until you've inspected the printed output below and found
# something that needs correcting. Format matches the dicts detect_sections
# returns, e.g.:
# SECTION_OVERRIDES = {
#     "110519": [
#         {"key": ("combined_totals_summary", None), "header_label": "combined_totals_summary",
#          "offices_seen": [...], "start_page": 1, "end_page": 4, "n_pages": 4},
#         ...
#     ]
# }
SECTION_OVERRIDES = {}

successful_ids = [d["election_id"] for d in download_log if d["status"] in ("downloaded", "already_downloaded")]

detected_sections = {}  # election_id -> list of section run dicts

for election_id in successful_ids:
    pdf_path = local_pdf_path(election_id)
    try:
        runs = detect_sections(pdf_path)
    except Exception as e:
        print(f"  [{election_id}] section detection failed: {e}")
        continue

    if election_id in SECTION_OVERRIDES:
        runs = SECTION_OVERRIDES[election_id]
        print(f"\n[{election_id}] using manual override ({len(runs)} runs)")

    detected_sections[election_id] = runs
    print_section_runs(runs, election_id)

n_unclassified_pages = sum(
    r["n_pages"] for runs in detected_sections.values() for r in runs
    if r["header_label"] is None
)
print(f"\n{'='*60}")
print(f"Total unclassified pages across all elections: {n_unclassified_pages}")
if n_unclassified_pages > 0:
    print("Some pages didn't match either known header pattern. This is")
    print("expected for older/different-era PDFs -- inspect those elections'")
    print("UNCLASSIFIED runs above before assuming the whole document parsed.")


## 5c. Sample first/middle/last page of each confirmed section, dedupe identical structures

For each confirmed section run, pulls the **first, middle, and last page**
of text. These three samples get reduced to a **structural fingerprint** —
not the raw text (which differs page to page just from the numbers/names
changing), but the *shape* of the page: column header line, count of
numeric columns, presence of a "Continued" marker, etc.

**This is the dedup step your workflow asked for.** Two section runs --
even from different elections, or different offices within the same
election -- that produce the same structural fingerprint are treated as
the same format and share one parser. This is the main lever against
error accumulation: instead of one parser per section run (which would
scale with document count and office count), you get one parser per
*distinct structural fingerprint*, however many runs share it.

The fingerprint function is intentionally conservative: two pages only
count as "the same format" if their column headers and structural shape
match. If you find this is over- or under-merging (treating genuinely
different formats as the same, or treating the same format as different
because of a minor wording difference), that's the function to adjust.


In [ ]:
def structural_fingerprint(text):
    """Reduces one page's text to a structure-only signature.

    pdfplumber extracts the real canvass tables with column headers
    fragmented across many short lines (e.g. "Total" / "Ballots" / "Cast"
    each on their own line, since the PDF stacks them vertically) -- a
    single-line "does this line contain 3+ header keywords" check does
    NOT find anything, confirmed against real extracted text. Instead,
    this finds the office line (e.g. "City of Houston, Mayor") and the
    first precinct-data row (a line starting with a 3-4 digit precinct
    number followed by a number), and joins everything between them.

    IMPORTANT: candidate names are collapsed to a single fixed <NAMES>
    placeholder, not just ignored and not counted. Two things were
    confirmed by direct test before landing on this: (1) keeping candidate
    names in the key makes every office in every election its own
    fingerprint (Mayor 2019 != Mayor 2023), which defeats the dedup goal
    entirely; (2) replacing names with a per-candidate WORD COUNT instead
    of a fixed token is still wrong, because word count varies with
    naming convention ("Bill King" vs "Sheila Jackson Lee") independent of
    how many candidates are actually in the race. So two pages are "the
    same format" when the office matches and the column SKELETON matches
    (Precinct, Election Ballots Cast, Total Ballots Cast, Registered
    Voters, Percent Turnout, <names>, Continued, Early Ballots Cast),
    regardless of how many candidates or what their names are -- the
    parser discovers the actual candidate count from each page at parse
    time, so it doesn't need to be part of the format key at all.
    """
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    office_idx = None
    first_data_idx = None
    office_re = re.compile(r"City of Houston,\s*(Mayor|Controller)|Council Member,?\s*(?:District\s+[A-K]|At[- ]Large Position\s+\d+)", re.IGNORECASE)
    data_row_re = re.compile(r"^\d{3,4}\s+\d")

    office_text = None
    for i, line in enumerate(lines):
        m = office_re.search(line)
        if office_idx is None and m:
            office_idx = i
            office_text = m.group(0)
        if office_idx is not None and first_data_idx is None and data_row_re.match(line):
            first_data_idx = i
            break

    if office_idx is not None and first_data_idx is not None:
        header_lines = lines[office_idx:first_data_idx]
        structural_kw = {"precinct", "election", "ballots", "cast", "total",
                          "registered", "voters", "percent", "turnout",
                          "continued", "early", "district"}
        # collapse ANY run of non-structural words to a single fixed
        # placeholder, not a word-count -- confirmed by direct test that
        # counting words in a name run is too noisy (e.g. "Bill King" vs
        # "John Whitmire" is 2 vs 2 words, but "Sheila Jackson Lee" alone
        # is 3 words for ONE candidate, so word-count varies with naming
        # convention, not with how many candidates are actually in the
        # race -- which is not a structural property worth keying on at
        # all here; the parser discovers the real candidate count from
        # the page at parse time regardless of what the fingerprint says)
        rebuilt = []
        in_name_run = False
        for raw_line in header_lines:
            for word in raw_line.split():
                cleaned = re.sub(r"[^a-zA-Z]", "", word).lower()
                if cleaned in structural_kw or cleaned == "":
                    in_name_run = False
                    rebuilt.append(word)
                else:
                    if not in_name_run:
                        rebuilt.append("<NAMES>")
                        in_name_run = True
        header_block = " ".join(rebuilt)
    else:
        header_block = None

    header_keywords = ["COUNT", "PERCENT", "COMBINED", "Canvass Report"]
    coarse_signal = tuple(sorted(kw for kw in header_keywords if kw.lower() in text.lower()))

    has_county_split = bool(re.search(r"Harris\s+County|Fort\s+Bend|Montgomery", text, re.IGNORECASE))

    return {
        "office_text": office_text,
        "header_block": header_block,
        "coarse_signal": coarse_signal,
        "has_county_split": has_county_split,
        "n_lines": len(lines),
    }

def fingerprint_key(fp):
    """Collapses a fingerprint dict to a hashable key for grouping. Two
    pages with the same key are treated as the same format and share one
    parser -- this is the lever against error accumulation: a parser
    group's size is bounded by the number of DISTINCT table shapes that
    exist (e.g. 'office with N candidate columns'), not by the number of
    election x office combinations, which would grow unboundedly."""
    if fp["header_block"] is not None:
        # normalize office_text so "City of Houston, Mayor" and a future
        # differently-capitalized variant of the same office still match
        normalized_office = re.sub(r"\s+", " ", (fp["office_text"] or "")).strip().lower()
        return ("detail", normalized_office, fp["header_block"])
    return ("coarse", fp["coarse_signal"], fp["has_county_split"])

def sample_and_fingerprint_run(pdf_path, run):
    """Pulls first/middle/last page text for one section run and computes
    a fingerprint for each. Returns (samples, is_internally_consistent)
    where is_internally_consistent is False if the first/middle/last pages
    of THIS SAME run don't share a fingerprint -- which would mean the
    run-detection in 5b grouped pages that don't actually belong together,
    and is worth double-checking before trusting any parser built from it.
    """
    start, end = run["start_page"], run["end_page"]
    mid = (start + end) // 2
    sample_pages = sorted(set([start, mid, end]))

    samples = []
    with pdfplumber.open(pdf_path) as pdf:
        for p in sample_pages:
            text = pdf.pages[p - 1].extract_text() or ""
            fp = structural_fingerprint(text)
            samples.append({"page": p, "text": text, "fingerprint": fp})

    keys = set(fingerprint_key(s["fingerprint"]) for s in samples)
    is_consistent = len(keys) == 1

    return samples, is_consistent

def dedupe_runs_across_elections(detected_sections, download_log):
    """Walks every confirmed run across every election, samples+fingerprints
    each, and groups runs sharing a fingerprint into one 'parser group'.
    Returns:
      parser_groups: {fingerprint_key: {"runs": [...], "sample": ...}}
      inconsistent_runs: list of (election_id, run) where first/mid/last
        pages disagreed -- flagged for manual review, not silently merged.
    """
    parser_groups = {}
    inconsistent_runs = []

    for election_id, runs in detected_sections.items():
        pdf_path = local_pdf_path(election_id)
        for run in runs:
            if run["header_label"] is None:
                continue  # unclassified pages handled separately
            samples, is_consistent = sample_and_fingerprint_run(pdf_path, run)
            if not is_consistent:
                inconsistent_runs.append((election_id, run, samples))
                continue

            key = fingerprint_key(samples[0]["fingerprint"])
            if key not in parser_groups:
                parser_groups[key] = {"runs": [], "sample_text": samples[0]["text"]}
            parser_groups[key]["runs"].append((election_id, run))

    return parser_groups, inconsistent_runs

parser_groups, inconsistent_runs = dedupe_runs_across_elections(detected_sections, download_log)

print(f"{len(parser_groups)} distinct structural fingerprint(s) found across all confirmed sections.")
print(f"{len(inconsistent_runs)} run(s) had inconsistent first/mid/last pages -- needs manual review.\n")

for i, (key, group) in enumerate(parser_groups.items()):
    n_elections = len(set(eid for eid, _ in group["runs"]))
    print(f"Fingerprint #{i}: {n_elections} election(s), {len(group['runs'])} run(s)")
    if key[0] == "detail":
        print(f"  type: detail   header_block: {str(key[1])[:100]}")
    else:
        print(f"  type: coarse   signal: {key[1]}   has_county_split: {key[2]}")
    print()

if inconsistent_runs:
    print("INCONSISTENT RUNS (first/mid/last pages didn't match within the same detected run):")
    for election_id, run, samples in inconsistent_runs:
        print(f"  [{election_id}] pages {run['start_page']}-{run['end_page']} ({run['header_label']})")
        for s in samples:
            print(f"    page {s['page']}: {str(s['fingerprint']['header_block'])[:80]}")


## 5d. One parser per distinct fingerprint, applied to every run that shares it

Each entry in `parser_groups` represents one distinct table shape. This
section writes one parser for the `canvass_report_detail` shape and one
for `combined_totals_summary`, applied to every run sharing that
fingerprint -- the direct payoff of the dedup step: one parsing function
per shape, not one per (election, office) pair.

**Important limitation, confirmed by direct testing before this was
written:** the real 2019 PDF's precinct-data rows, when extracted as
plain text via `page.extract_text()`, collapse into a single
whitespace-joined line of numbers with no reliable separator between
"Total Ballots Cast", "Registered Voters", and "Percent Turnout" --
multiple candidate column-count mappings are consistent with the same
token sequence, and a sanity check (`election_ballots + early_ballots ==
total_ballots`, which should hold by definition) **did not validate**
against the most plausible left-to-right mapping. Rather than ship a
column mapping that might be silently wrong -- the exact failure mode
this whole redesign exists to prevent -- this parser uses
**`page.extract_table()`**, which reads the PDF's actual drawn cell
boundaries/x-coordinates instead of inferring structure from text order.
This needs the real PDF file and a live `pdfplumber` session to test,
which isn't available outside Colab, so:

**Action needed from you:** run this section, then check the printed
`COLUMN MAPPING CHECK` output for the first parsed election. It re-derives
the same `election + early == total` identity from the *actual* extracted
table and tells you whether it held. If it didn't, the column mapping
below needs adjusting before trusting any vote counts -- don't proceed to
Section 6 (reconciliation) until this check passes.


In [ ]:
def parse_canvass_detail_page(page, page_idx, election_id, election_year, office_text):
    """Parses one page of the canvass_report_detail shape using
    extract_table() instead of extract_text() + regex.

    extract_table() reads the PDF's actual drawn cell boundaries, so each
    table row comes back as a LIST OF CELLS already split at the right
    points -- this sidesteps the column-mapping ambiguity that showed up
    when trying to infer structure from whitespace-joined text (a direct
    test of that approach failed a basic sanity check: election_ballots +
    early_ballots should equal total_ballots, and it didn't, with the most
    plausible left-to-right token mapping).

    Expected cell layout per data row, left to right:
      [precinct, election_ballots, total_ballots, registered_voters,
       pct_turnout, <candidate_1>, <candidate_2>, ..., <candidate_N>,
       early_ballots]

    Falls back to flagging the row with parse_notes if extract_table()
    returns nothing usable (e.g. a page with no visible grid lines), so a
    bad page surfaces in QA rather than silently producing zero rows.
    """
    vote_rows = []
    summary_rows = []

    tables = page.extract_table()
    if not tables:
        summary_rows.append({
            "election_id": election_id, "election_year": election_year,
            "page_number": page_idx + 1, "office": office_text,
            "label": "NO_TABLE_EXTRACTED", "parse_notes": "extract_table_returned_none",
        })
        return {"vote_rows": vote_rows, "summary_rows": summary_rows}

    # find candidate names from the header row(s) at the top of this page's
    # table -- the header row is whichever row contains "Precinct" as a cell
    header_row_idx = None
    for i, row in enumerate(tables):
        if row and any(cell and "precinct" in str(cell).lower() for cell in row):
            header_row_idx = i
            break

    candidate_names = []
    if header_row_idx is not None:
        header_row = tables[header_row_idx]
        # cells between the known fixed columns and "Continued"/"Early" are
        # candidate names; fixed-column labels are excluded by keyword match
        fixed_labels = {"precinct", "election", "ballots", "cast", "total",
                         "registered", "voters", "percent", "turnout",
                         "continued", "early"}
        for cell in header_row:
            if not cell:
                continue
            cell_clean = str(cell).strip()
            if not cell_clean:
                continue
            words = cell_clean.lower().split()
            if all(w.strip(".,") in fixed_labels for w in words):
                continue
            candidate_names.append(cell_clean.replace("\n", " "))

    data_rows = [r for r in tables if r and r[0] and re.match(r"^\d{3,4}$", str(r[0]).strip())]

    for row in data_rows:
        cells = [str(c).replace(",", "").strip() if c else "" for c in row]
        if len(cells) < 6:
            continue
        precinct = cells[0]
        try:
            election_ballots = int(cells[1])
            total_ballots = int(cells[2])
            registered_voters = int(cells[3])
        except (ValueError, IndexError):
            summary_rows.append({
                "election_id": election_id, "election_year": election_year,
                "page_number": page_idx + 1, "office": office_text,
                "label": f"unparseable_row_precinct_{precinct}",
                "parse_notes": "fixed_column_not_numeric",
            })
            continue

        pct_turnout = cells[4] if len(cells) > 4 else None
        candidate_cells = cells[5:-1] if len(cells) > 6 else []
        early_ballots = cells[-1] if len(cells) > 5 else None

        for ci, cand_name in enumerate(candidate_names):
            vote_count = candidate_cells[ci] if ci < len(candidate_cells) else None
            vote_rows.append({
                "election_id": election_id, "election_year": election_year,
                "page_number": page_idx + 1, "office": office_text,
                "precinct": precinct, "candidate": cand_name,
                "election_ballots": election_ballots, "total_ballots": total_ballots,
                "registered_voters": registered_voters, "pct_turnout": pct_turnout,
                "votes": vote_count, "early_ballots": early_ballots,
                "parse_notes": "" if ci < len(candidate_cells) else "candidate_count_mismatch",
            })

    return {"vote_rows": vote_rows, "summary_rows": summary_rows}


def check_column_mapping(vote_rows_df):
    """Sanity check: for rows where we have numeric election_ballots and
    early_ballots, does election_ballots + early_ballots == total_ballots?
    This identity should hold by definition (every ballot is either an
    election-day ballot or an early/absentee ballot). If it doesn't hold
    for most rows, the column mapping in parse_canvass_detail_page is
    wrong and must be fixed before trusting any vote counts.
    """
    if vote_rows_df.empty:
        print("  (no vote rows to check)")
        return

    sample = vote_rows_df.drop_duplicates(subset=["election_id", "precinct", "office"]).head(200)
    n_checked = 0
    n_passed = 0
    for _, row in sample.iterrows():
        try:
            eb = int(row["election_ballots"])
            early = int(row["early_ballots"])
            tb = int(row["total_ballots"])
        except (ValueError, TypeError):
            continue
        n_checked += 1
        if eb + early == tb:
            n_passed += 1

    print(f"COLUMN MAPPING CHECK: {n_passed}/{n_checked} sampled rows satisfy "
          f"election_ballots + early_ballots == total_ballots")
    if n_checked > 0 and n_passed / n_checked < 0.9:
        print("  *** MOST ROWS FAILED THIS CHECK. Column mapping is likely wrong. ***")
        print("  Do not trust vote counts until this is fixed.")
    elif n_checked == 0:
        print("  (no rows had all three fields as valid integers -- check parse_notes)")
    else:
        print("  Mapping looks consistent.")


def parse_combined_totals_page(page, page_idx, election_id, election_year):
    """Parses one page of the combined_totals_summary shape (the
    citywide office-by-office results that appear before the precinct
    detail section). This section's layout -- candidate name, count,
    percent, repeated per office, with a later county-split repeat of the
    same offices -- is coarser and less uniformly tabular than the
    precinct detail section, so this parser is intentionally limited to
    extracting (office, candidate, count, percent) tuples where they can
    be confidently identified, and flags anything ambiguous rather than
    guessing.
    """
    summary_rows = []
    text = page.extract_text() or ""
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    current_office = None
    office_re = re.compile(r"^(MAYOR|CONTROLLER|COUNCIL MEMBER.*?)(?:\s+COUNT\s+PERCENT)?$", re.IGNORECASE)

    for line in lines:
        m = office_re.match(line)
        if m:
            current_office = m.group(1).strip()
            continue
        summary_rows.append({
            "election_id": election_id, "election_year": election_year,
            "page_number": page_idx + 1, "office": current_office,
            "raw_line": line[:150], "parse_notes": "combined_totals_unstructured_capture",
        })

    return {"vote_rows": [], "summary_rows": summary_rows}


## 5e. Run the parsers across every confirmed run, then check the column mapping

This applies `parse_canvass_detail_page` / `parse_combined_totals_page` to
every page in every run from `parser_groups`, and immediately runs
`check_column_mapping` on the result.

**Stop here and fix the parser before continuing if the check fails.**
Section 6 (reconciliation against the document's own printed totals) is
the next safety net, but it can't catch a column-mapping bug that shifts
every value over by one position consistently -- that kind of error can
still sum to the right total by coincidence in some cases. The
`election_ballots + early_ballots == total_ballots` identity checked here
is a more basic, harder-to-fool sanity check that should pass for nearly
every row if the columns are mapped correctly.


In [ ]:
all_vote_rows = []
all_summary_rows = []
parse_errors = []

for election_id, runs in detected_sections.items():
    pdf_path = local_pdf_path(election_id)
    election_year = next(d["election_year"] for d in download_log if d["election_id"] == election_id)

    with pdfplumber.open(pdf_path) as pdf:
        for run in runs:
            if run["header_label"] is None:
                continue
            for page_num in range(run["start_page"], run["end_page"] + 1):
                page = pdf.pages[page_num - 1]
                try:
                    if run["header_label"] == "canvass_report_detail":
                        office_text = run.get("office_line") or (run["offices_seen"][0] if run["offices_seen"] else None)
                        result = parse_canvass_detail_page(page, page_num - 1, election_id, election_year, office_text)
                    elif run["header_label"] == "combined_totals_summary":
                        result = parse_combined_totals_page(page, page_num - 1, election_id, election_year)
                    else:
                        continue
                except Exception as e:
                    parse_errors.append({"election_id": election_id, "page": page_num, "error": str(e)})
                    continue

                all_vote_rows.extend(result["vote_rows"])
                all_summary_rows.extend(result["summary_rows"])

df_votes = pd.DataFrame(all_vote_rows) if all_vote_rows else pd.DataFrame()
df_summary_raw = pd.DataFrame(all_summary_rows) if all_summary_rows else pd.DataFrame()

print(f"df_votes: {len(df_votes)} rows")
print(f"df_summary_raw: {len(df_summary_raw)} rows")
print(f"{len(parse_errors)} page(s) raised parser exceptions")
if parse_errors:
    for e in parse_errors[:10]:
        print(f"  [{e['election_id']}] page {e['page']}: {e['error']}")

print()
check_column_mapping(df_votes)


## 6. Reconciliation: parsed totals vs. the document's own printed totals

This is the parser-agnostic tripwire discussed earlier: it doesn't matter
how a vote count was produced -- if the precinct-level rows for an office
don't sum to the official total that the *same PDF* prints elsewhere
(in the combined_totals_summary section, or in a per-office "Totals" row
within canvass_report_detail), something is wrong, and it's wrong
regardless of which parser produced the numbers.

This check is what should give you actual confidence in the output,
more than any individual parser's internal logic -- it's checking the
parser's output against the document's own stated truth, not against
another guess.

Any office where the sums don't match gets flagged in `reconciliation_failures`
rather than silently accepted.


In [ ]:
def normalize_office_name(office_text):
    """The combined_totals_summary section labels offices as 'MAYOR' /
    'COUNCIL MEMBER DISTRICT A', while canvass_report_detail labels them
    'City of Houston, Mayor' / 'Council Member, District A' -- confirmed
    these are genuinely different strings that would NOT match on a plain
    equality merge. This reduces both to a comparable key: lowercase,
    strip 'city of houston,' prefix, collapse punctuation/whitespace.
    """
    if office_text is None:
        return None
    t = str(office_text).lower()
    t = t.replace("city of houston,", "").replace(",", " ")
    t = re.sub(r"\s+", " ", t).strip()
    return t


def extract_office_totals_from_summary(df_summary_raw):
    """Pulls (election_id, office, total_votes) tuples from the
    combined_totals_summary section's raw captured lines.

    Confirmed against real extracted text: this section does NOT
    interleave a count and its percent on the same line. Instead,
    pdfplumber extracts a full run of count-lines for an office, followed
    by a SEPARATE run of percent-lines, e.g.:
        1415
        7182
        ...
        15808       <- grand total vote count for this office
        9.0%
        45.4%
        ...
        100.0%      <- grand total percent, confirms 15808 was the total
    A same-line "NUMBER 100.0%" regex (the first version of this function,
    before checking real text) never matches this layout at all -- it was
    written against an assumption, not the actual extracted structure.

    This version: for each office, walks its captured raw_lines in order;
    when a line is exactly "100.0%", the grand total is the bare-number
    line immediately preceding the start of that percent-run (i.e. the
    last count-line before percent-lines begin).
    """
    totals = []
    if df_summary_raw.empty:
        return pd.DataFrame(columns=["election_id", "office", "office_normalized", "printed_total"])

    bare_number_re = re.compile(r"^[\d,]+$")
    percent_re = re.compile(r"^\d+(\.\d+)?%$")

    for (election_id, office), group in df_summary_raw.groupby(["election_id", "office"]):
        lines = [str(row.get("raw_line", "") or "").strip() for _, row in group.iterrows()]

        last_bare_number = None
        in_percent_run = False
        for line in lines:
            if bare_number_re.match(line):
                if not in_percent_run:
                    last_bare_number = line
                # a bare number appearing AFTER percents started would be a
                # new office's counts beginning -- reset
                else:
                    in_percent_run = False
                    last_bare_number = line
            elif percent_re.match(line):
                if line == "100.0%" and last_bare_number is not None:
                    totals.append({
                        "election_id": election_id,
                        "office": office,
                        "office_normalized": normalize_office_name(office),
                        "printed_total": int(last_bare_number.replace(",", "")),
                    })
                in_percent_run = True

    return pd.DataFrame(totals)


def reconcile_parsed_vs_printed(df_votes, df_printed_totals):
    """For each (election_id, office) with both parsed vote rows and a
    printed total, sums the parsed votes and compares. Returns a
    DataFrame of results with a pass/fail flag -- this is independent of
    which parser produced df_votes, so it catches mapping errors that a
    parser's own internal logic could not catch on its own.

    Merges on a NORMALIZED office key, not the raw office string --
    confirmed the two sections label offices differently ('MAYOR' vs
    'City of Houston, Mayor'), so a plain-string merge would silently
    produce zero matches and look like "nothing to reconcile" instead of
    surfacing a real mismatch to fix.
    """
    if df_votes.empty or df_printed_totals.empty:
        return pd.DataFrame()

    df_votes_numeric = df_votes.copy()
    df_votes_numeric["votes_numeric"] = pd.to_numeric(
        df_votes_numeric["votes"].astype(str).str.replace(",", ""), errors="coerce"
    )
    df_votes_numeric["office_normalized"] = df_votes_numeric["office"].apply(normalize_office_name)

    parsed_sums = (
        df_votes_numeric.groupby(["election_id", "office_normalized"])["votes_numeric"]
        .sum()
        .reset_index()
        .rename(columns={"votes_numeric": "parsed_total"})
    )

    merged = df_printed_totals.merge(parsed_sums, on=["election_id", "office_normalized"], how="left")
    merged["parsed_total"] = merged["parsed_total"].fillna(0)
    merged["difference"] = merged["parsed_total"] - merged["printed_total"]
    merged["passes"] = merged["difference"].abs() <= (merged["printed_total"] * 0.001).clip(lower=1)

    return merged


df_printed_totals = extract_office_totals_from_summary(df_summary_raw)
print(f"Found {len(df_printed_totals)} printed office total(s) in the combined-totals section.")

reconciliation = reconcile_parsed_vs_printed(df_votes, df_printed_totals)

if reconciliation.empty:
    print("\nNo reconciliation possible yet -- check that both df_votes and")
    print("df_printed_totals have rows before trusting any parsed numbers.")
else:
    n_pass = reconciliation["passes"].sum()
    n_total = len(reconciliation)
    print(f"\n{n_pass}/{n_total} office(s) reconcile within 0.1% of the printed total.")

    reconciliation_failures = reconciliation[~reconciliation["passes"]]
    if not reconciliation_failures.empty:
        print("\nFAILURES (parsed total does not match printed total):")
        print(reconciliation_failures[["election_id", "office", "printed_total", "parsed_total", "difference"]].to_string(index=False))
        print("\nDo not treat df_votes as ground truth for these offices until this is resolved.")
    else:
        print("All reconciled offices match. This is real evidence the parser is correct")
        print("for these offices -- it is not just internally consistent, it matches an")
        print("independent total printed elsewhere in the same document.")


## 7. Summary of the combined dataset

`df_votes` and `df_summary_raw` (built in Section 5e) are already combined
across every election that was parsed -- the section-detection and
fingerprint-dedup approach means there's no separate per-election
DataFrame to concatenate; every run, regardless of which election it came
from, fed into the same `df_votes` via its shared parser.

`df_all_votes` / `df_all_summary` are kept as aliases for these two so the
Export section below doesn't need to change names if you're used to the
earlier version of this notebook.


In [ ]:
df_all_votes = df_votes
df_all_summary = df_summary_raw

n_elections_votes = df_all_votes["election_id"].nunique() if not df_all_votes.empty else 0
n_elections_summary = df_all_summary["election_id"].nunique() if not df_all_summary.empty else 0

print(f"df_all_votes:   {len(df_all_votes)} rows across {n_elections_votes} elections")
print(f"df_all_summary: {len(df_all_summary)} rows across {n_elections_summary} elections")

df_all_votes.head(20)


## 8. Export

Saves:
- one CSV per election's vote rows + summary rows under `/content/coh_canvass/exports/<election_id>/`
- combined `df_all_votes.csv` / `df_all_summary.csv` / `reconciliation.csv`
- a single XLSX with one sheet per election (votes only, for quick browsing), plus combined and reconciliation sheets


In [ ]:
EXPORT_DIR = os.path.join(WORKDIR, "exports")
os.makedirs(EXPORT_DIR, exist_ok=True)

election_ids_present = sorted(set(df_all_votes["election_id"]) | set(df_all_summary["election_id"])) if not df_all_votes.empty or not df_all_summary.empty else []

for election_id in election_ids_present:
    election_dir = os.path.join(EXPORT_DIR, election_id)
    os.makedirs(election_dir, exist_ok=True)
    votes_slice = df_all_votes[df_all_votes["election_id"] == election_id] if not df_all_votes.empty else pd.DataFrame()
    summary_slice = df_all_summary[df_all_summary["election_id"] == election_id] if not df_all_summary.empty else pd.DataFrame()
    if not votes_slice.empty:
        votes_slice.to_csv(os.path.join(election_dir, "votes.csv"), index=False)
    if not summary_slice.empty:
        summary_slice.to_csv(os.path.join(election_dir, "summary.csv"), index=False)

df_all_votes.to_csv(os.path.join(EXPORT_DIR, "df_all_votes.csv"), index=False)
df_all_summary.to_csv(os.path.join(EXPORT_DIR, "df_all_summary.csv"), index=False)
if not reconciliation.empty:
    reconciliation.to_csv(os.path.join(EXPORT_DIR, "reconciliation.csv"), index=False)

xlsx_path = os.path.join(EXPORT_DIR, "coh_canvass_combined.xlsx")
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    df_all_votes.to_excel(writer, sheet_name="all_votes", index=False)
    df_all_summary.to_excel(writer, sheet_name="all_summary", index=False)
    if not reconciliation.empty:
        reconciliation.to_excel(writer, sheet_name="reconciliation", index=False)
    for election_id in election_ids_present:
        votes_slice = df_all_votes[df_all_votes["election_id"] == election_id] if not df_all_votes.empty else pd.DataFrame()
        if not votes_slice.empty:
            # sheet names capped at 31 chars by Excel
            sheet_name = f"v_{election_id}"[:31]
            votes_slice.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Per-election CSVs written under {EXPORT_DIR}/<election_id>/")
print("Combined CSVs: df_all_votes.csv, df_all_summary.csv, reconciliation.csv")
print(f"Combined workbook: {xlsx_path}")


## 9. (Optional) Download exports to your machine


In [ ]:
try:
    from google.colab import files
    import shutil
    zip_path = "/content/coh_canvass_exports"
    shutil.make_archive(zip_path, "zip", EXPORT_DIR)
    files.download(zip_path + ".zip")
except ImportError:
    print("Not running in Colab -- exports are already on disk at:", EXPORT_DIR)


## Notes on extending this

- **`parse_notes` columns are your QA trail.** Every row that came from a
  heuristic guess or a fallback path is flagged. Filter on
  `parse_notes != ""` to find rows worth spot-checking against the source
  PDF before treating this as ground truth.
- **The column-mapping self-check (Section 5e) and the reconciliation
  check (Section 6) are the two most important things to look at before
  trusting any output.** Neither is optional. The column-mapping check
  catches a parser that's systematically reading the wrong cell as the
  wrong field; the reconciliation check catches that *and* anything else,
  by comparing against a number the document prints independently of
  whatever the parser did. If either fails, fix the parser before using
  `df_votes` for anything.
- **A new table shape showing up?** It'll appear in the Section 5c output
  as its own fingerprint group with no matching parser in Section 5d,
  rather than getting silently absorbed into an existing parser or
  silently dropped. Add a new `parse_*_page` function for that shape,
  branch to it in Section 5e's dispatch logic, and re-run from Section 5e
  onward — Sections 1-5c (download, section-detection, sampling/dedup)
  don't need to re-run.
- **Section breaks that look wrong?** Check the Section 5b output first.
  If a run's page range or `offices_seen` is off for one specific
  election, use `SECTION_OVERRIDES` rather than editing the detection
  regexes (which would affect every election). If it's wrong across
  *every* election, the regexes in Section 5 (`SECTION_HEADER_PATTERNS`,
  `OFFICE_LINE_RE`) need fixing instead.
- **Why fingerprints ignore candidate names:** confirmed by direct test
  that keeping names in the fingerprint key makes every office in every
  election its own format (since candidates differ year to year), which
  defeats the entire point of deduping. The fingerprint only tracks the
  office and the surrounding column skeleton; the actual candidate count
  and names are read from each page at parse time, not baked into the
  format key.
- **This is read-only against the live site** — it never POSTs/submits
  anything, only follows public `.pdf` links from the index page.
